# Prequential Evaluation capyMOA

In [ ]:
from capymoa.anomaly import OnlineIsolationForest
from capymoa.anomaly import HalfSpaceTrees
from capymoa.evaluation.visualization import plot_windowed_results
from capymoa.evaluation import prequential_evaluation_anomaly
from src.Data.Processor import DataStreamProcessor
import pandas as pd

df = pd.read_csv('data/15k/Consistência/Consistência_2000.csv')

processor = DataStreamProcessor(logging=True)

stream, targets, features = processor.create_stream(
    df=df, 
    target_label_col='Label', 
    binary_label=False, 
    normalize_method="MinMaxScaler", 
    threshold_var=0.005,
    threshold_corr=None,
    top_n_features=None,
    return_stream=True,
    extra_ignore_cols=['Source IP', 'Source Port','Destination IP','Destination Port','Protocol'],
    imputation_method='mediana'
)

oif = OnlineIsolationForest(schema=stream.get_schema())
hst = HalfSpaceTrees(schema=stream.get_schema())

results_hst = prequential_evaluation_anomaly(
    stream=stream, learner=hst, window_size=200
)

results_oif = prequential_evaluation_anomaly(
    stream=stream, learner=oif, window_size=200
)

In [ ]:
# print("Half-Space Trees Results:")
# print(f"AUC: {results_hst.auc()}")
# print(f"Accuracy: {results_hst.accuracy()}")
# print(f"Recall: {results_hst.recall()}")

# print("Online Isolation Forest Results:")
# print(f"AUC: {results_oif.auc()}")
# print(f"Accuracy: {results_oif.accuracy()}")
# print(f"Recall: {results_oif.recall()}")

display(results_hst.windowed.metrics_per_window())
plot_windowed_results(results_hst, results_oif, metric="Accuracy", save_only=False)
plot_windowed_results(results_hst, results_oif, metric="Recall", save_only=False)
plot_windowed_results(results_hst, results_oif, metric="auc", save_only=False)

# Execução Pipelines

## Otimização Optuna

### Melhores Features

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from src.Data.Processor import DataStreamProcessor

# Definição dos datasets
categorias = ['Consistência', 'Generalização', 'Adaptação', 'Recorrência']
tamanhos = ['25', '200', '1000']
datasets = [f'data/15k/{cat}/{cat}_{tam}.csv' for cat in categorias for tam in tamanhos]

# Dicionário/Counter para armazenar a contagem global das features
contagem_features = Counter()

for dataset_path in datasets:
    print(f"-> Analisando: {dataset_path}")
    
    try:
        df = pd.read_csv(dataset_path)
    except FileNotFoundError:
        print(f"   [!] Arquivo não encontrado. Pulando...")
        continue
    
    processor = DataStreamProcessor(logging=False)
    
    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=0.85,
        top_n_features=25,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
        imputation_method='mediana'
    )
    
    # Adiciona as 25 features deste cenário ao nosso contador global
    contagem_features.update(features)

# Pega as 25 features que mais apareceram somando todos os cenários
top_25_globais = contagem_features.most_common(25)
nomes_features = [f[0] for f in top_25_globais]

print("\nfeatures = [")
for feat in nomes_features:
    print(f"    '{feat}',")
print("]")

### Otimização dos Modelos

In [ ]:
%load_ext autoreload
%autoreload 2

from src.Anomaly.Optimizer import AnomalyOptunaOptimizer
from src.Data.Processor import DataStreamProcessor
import pandas as pd

# Definição dos datasets
categorias = ['Consistência', 'Generalização', 'Adaptação', 'Recorrência']
tamanhos = ['25', '200', '1000']
datasets = [f'data/15k/{cat}/{cat}_{tam}.csv' for cat in categorias for tam in tamanhos]

features = [
    'Fwd Packet Length Min',
    'Total Fwd Packets',
    'Fwd Packet Length Max',
    'Packet Length Variance',
    'Init_Win_bytes_forward',
    'Flow IAT Mean',
    'Fwd Packets/s',
    'Fwd Packet Length Std',
    'Flow Duration',
    'Total Backward Packets',
    'URG Flag Count',
    'Init_Win_bytes_backward',
    'Flow IAT Min',
    'Bwd Packets/s',
    'Bwd IAT Mean',
    'Down/Up Ratio',
    'Bwd IAT Min',
    'Bwd Packet Length Mean',
    'Bwd Packet Length Max',
    'Fwd Header Length',
    'Total Length of Fwd Packets',
    'ACK Flag Count',
    'Active Mean',
    'Fwd Packet Length Mean',
    'Fwd PSH Flags',
]

for dataset_path in datasets:
    print(f"\nIniciando otimização para: {dataset_path}")
    
    df = pd.read_csv(dataset_path)
    
    processor = DataStreamProcessor(logging=False)
    
    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol'],
        imputation_method='mediana'
    )
    
    optimizer = AnomalyOptunaOptimizer(
        stream=stream,
        n_trials=5,
        discretization_threshold='params',
        target_class='macro',
        target_class_pass=0,
        target_names=targets
    )

    # melhor_hst = optimizer.optimize('HST')
    # melhor_oif = optimizer.optimize('OIF')
    melhor_aif = optimizer.optimize('AIF')
    # melhor_aif = optimizer.optimize('AE', warmup_instances=1000)
    # melhor_rrcf = optimizer.optimize('RRCF')

## Resultados

In [ ]:
%load_ext autoreload
%autoreload 2

from src.Anomaly.Pipeline import AnomalyExperimentRunner
from src.Anomaly.Models import get_anomaly_models
from src.Data.Processor import DataStreamProcessor
import pandas as pd

categorias = ['Consistência', 'Generalização', 'Adaptação', 'Recorrência']
tamanhos = ['25', '200', '1000']
datasets = [f'data/15k/{cat}/{cat}_{tam}.csv' for cat in categorias for tam in tamanhos]

features = [
    'Fwd Packet Length Min',
    'Total Fwd Packets',
    'Fwd Packet Length Max',
    'Packet Length Variance',
    'Init_Win_bytes_forward',
    'Flow IAT Mean',
    'Fwd Packets/s',
    'Fwd Packet Length Std',
    'Flow Duration',
    'Total Backward Packets',
    'URG Flag Count',
    'Init_Win_bytes_backward',
    'Flow IAT Min',
    'Bwd Packets/s',
    'Bwd IAT Mean',
    'Down/Up Ratio',
    'Bwd IAT Min',
    'Bwd Packet Length Mean',
    'Bwd Packet Length Max',
    'Fwd Header Length',
    'Total Length of Fwd Packets',
    'ACK Flag Count',
    'Active Mean',
    'Fwd Packet Length Mean',
    'Fwd PSH Flags',
]

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')
    print(f"\nIniciando treinamento para: {nome_experimento}")
    
    df = pd.read_csv(dataset_path)
    
    processor = DataStreamProcessor(logging=False, selected_features=features)
    
    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
        imputation_method='mediana'
    )
    
    algoritmos = get_anomaly_models(
        stream.get_schema(),
        selected_models=['AIF']
    )
    
    runner = AnomalyExperimentRunner(target_names=targets)
    
    runner._run_anomaly_evaluation(
        stream,
        algorithms=algoritmos,
        window_size=100,
        warmup_instances=0,
        title=nome_experimento,
        target_class='macro',
        target_class_pass=0,
        threshold=0.48
    )